# Прогноз долей категорий через две RandomForest-модели

Идея пайплайна:
1. `rf_total_model` прогнозирует общий спрос на следующий час.
2. `rf_category_model` прогнозирует спрос каждой категории на тот же следующий час.
3. Категорийные прогнозы переводятся в проценты.
4. Общий прогноз распределяется по категориям в этих процентах.

Если сохраненные модели уже есть рядом с ноутбуком, они загружаются. Если файлов нет, модели обучаются и сохраняются.

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (15, 6)
plt.rcParams["axes.grid"] = True

## Настройки

In [ ]:
DATA_PATH = "df_with_cat.csv"

# Города берем в оригинальном написании из df_with_cat.csv.
CITY_NAMES_ORIGINAL = [
    "İstanbul",
    "Kocaeli",
    "Bursa",
    "Ankara",
    "Antalya",
    "Konya",
    "İzmir",
    "Yozgat",
    "Kayseri",
    "Adana",
    "Gaziantep",
    "Diyarbakır",
]
CITY_NAME = CITY_NAMES_ORIGINAL[0]

TEST_SIZE = 0.20
RANDOM_STATE = 42
RF_TREES = 500

TOTAL_MODEL_PATH = Path("rf_total_model.joblib")
CATEGORY_MODEL_PATH = Path("rf_category_model.joblib")
USE_SAVED_MODELS = True

## Загрузка данных

In [ ]:
def resolve_data_path(path: str | Path) -> Path:
    path = Path(path)
    candidates = [path, Path("..") / path]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Не найден {path}. Проверены пути: "
        + ", ".join(str(candidate) for candidate in candidates)
    )


def load_events(path: str | Path) -> pd.DataFrame:
    events = pd.read_csv(resolve_data_path(path))
    events["model_response_timestamp"] = pd.to_datetime(
        events["model_response_timestamp"],
        unit="s",
    )
    events["date_hour"] = events["model_response_timestamp"].dt.floor("h")
    events["category"] = events["category"].fillna("unknown").astype(str)
    return events


events = load_events(DATA_PATH)

missing_cities = sorted(set(CITY_NAMES_ORIGINAL) - set(events["name"].unique()))
if missing_cities:
    raise ValueError(f"В CSV не найдены города: {missing_cities}")

display(
    events.loc[events["name"].isin(CITY_NAMES_ORIGINAL), "name"]
    .value_counts()
    .reindex(CITY_NAMES_ORIGINAL)
    .rename("rows")
    .reset_index(names="city")
)

events.head()

## Подготовка признаков для общего спроса

In [ ]:
def build_total_base(events: pd.DataFrame, city_name: str) -> pd.DataFrame:
    hourly = (
        events.groupby(["name", "date_hour"])
        .size()
        .reset_index(name="count")
    )

    city_hourly = hourly.loc[hourly["name"] == city_name, ["date_hour", "count"]]
    all_hours = pd.date_range(
        city_hourly["date_hour"].min(),
        city_hourly["date_hour"].max(),
        freq="h",
    )

    base_df = (
        city_hourly.set_index("date_hour")
        .reindex(all_hours, fill_value=0)
        .rename_axis("date_hour")
        .reset_index()
    )
    base_df.insert(0, "name", city_name)
    return base_df.sort_values("date_hour").reset_index(drop=True)


def add_total_time_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values("date_hour").reset_index(drop=True).copy()
    result["hour"] = result["date_hour"].dt.hour
    result["day"] = result["date_hour"].dt.day
    result["month"] = result["date_hour"].dt.month
    result["day_of_week"] = result["date_hour"].dt.dayofweek
    result["is_weekend"] = result["day_of_week"].isin([5, 6]).astype(int)
    result["hour_sin"] = np.sin(2 * np.pi * result["hour"] / 24)
    result["hour_cos"] = np.cos(2 * np.pi * result["hour"] / 24)

    for day in range(7):
        result[f"day_of_week_{day}"] = (result["day_of_week"] == day).astype(int)

    return result


def add_total_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values("date_hour").reset_index(drop=True).copy()
    result["lag_1"] = result["count"].shift(1)
    result["lag_2"] = result["count"].shift(2)
    result["lag_24"] = result["count"].shift(24)
    result["rolling_mean_3"] = result["count"].shift(1).rolling(3).mean()
    result["rolling_mean_6"] = result["count"].shift(1).rolling(6).mean()
    result["rolling_mean_24"] = result["count"].shift(1).rolling(24).mean()
    return result


def build_total_features(df: pd.DataFrame) -> pd.DataFrame:
    return add_total_lag_features(add_total_time_features(df))


total_base_df = build_total_base(events, CITY_NAME)
total_featured_df = build_total_features(total_base_df)
total_featured_df["predict_1h"] = total_featured_df["count"].shift(-1)
total_featured_df["target_hour"] = total_featured_df["date_hour"] + pd.Timedelta(hours=1)

total_excluded_dates = total_featured_df["month"].eq(2) & total_featured_df["day"].isin([14, 15, 25])
total_featured_df = total_featured_df.loc[~total_excluded_dates].reset_index(drop=True)

day_columns = [f"day_of_week_{day}" for day in range(7)]
total_feature_cols = [
    "count",
    "hour_sin",
    "hour_cos",
    "lag_1",
    "lag_2",
    "lag_24",
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_mean_24",
    "is_weekend",
    *day_columns,
]

total_model_df = total_featured_df.dropna(
    subset=total_feature_cols + ["predict_1h"]
).reset_index(drop=True)

display(total_model_df[["date_hour", "target_hour", "count", "predict_1h"]].head())

## Подготовка признаков для категорий

In [ ]:
def make_ohe_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_category_base(events: pd.DataFrame, city_name: str) -> pd.DataFrame:
    hourly = (
        events.groupby(["name", "category", "date_hour"])
        .size()
        .reset_index(name="count")
    )

    city_hourly = hourly.loc[
        hourly["name"] == city_name,
        ["date_hour", "category", "count"],
    ]

    all_hours = pd.date_range(
        city_hourly["date_hour"].min(),
        city_hourly["date_hour"].max(),
        freq="h",
    )
    all_categories = sorted(city_hourly["category"].unique())

    full_index = pd.MultiIndex.from_product(
        [all_hours, all_categories],
        names=["date_hour", "category"],
    )

    base_df = (
        city_hourly.set_index(["date_hour", "category"])
        .reindex(full_index, fill_value=0)
        .reset_index()
    )
    base_df.insert(0, "name", city_name)
    base_df = base_df.sort_values(["date_hour", "category"]).reset_index(drop=True)

    encoder = make_ohe_encoder()
    category_ohe = encoder.fit_transform(base_df[["category"]])
    category_ohe_df = pd.DataFrame(
        category_ohe,
        columns=encoder.get_feature_names_out(["category"]),
        index=base_df.index,
    )

    return pd.concat([base_df, category_ohe_df], axis=1)


def add_category_time_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values(["category", "date_hour"]).reset_index(drop=True).copy()
    result["hour"] = result["date_hour"].dt.hour
    result["day"] = result["date_hour"].dt.day
    result["month"] = result["date_hour"].dt.month
    result["day_of_week"] = result["date_hour"].dt.dayofweek
    result["is_weekend"] = result["day_of_week"].isin([5, 6]).astype(int)
    result["hour_sin"] = np.sin(2 * np.pi * result["hour"] / 24)
    result["hour_cos"] = np.cos(2 * np.pi * result["hour"] / 24)

    for day in range(7):
        result[f"day_of_week_{day}"] = (result["day_of_week"] == day).astype(int)

    return result


def add_category_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values(["category", "date_hour"]).reset_index(drop=True).copy()
    group = result.groupby("category")["count"]
    result["lag_1"] = group.shift(1)
    result["lag_2"] = group.shift(2)
    result["lag_24"] = group.shift(24)
    result["rolling_mean_3"] = group.transform(lambda x: x.shift(1).rolling(3).mean())
    result["rolling_mean_6"] = group.transform(lambda x: x.shift(1).rolling(6).mean())
    result["rolling_mean_24"] = group.transform(lambda x: x.shift(1).rolling(24).mean())
    return result


def build_category_features(df: pd.DataFrame) -> pd.DataFrame:
    return add_category_lag_features(add_category_time_features(df))


category_base_df = build_category_base(events, CITY_NAME)
category_featured_df = build_category_features(category_base_df)
category_featured_df["predict_1h"] = category_featured_df.groupby("category")["count"].shift(-1)
category_featured_df["target_hour"] = category_featured_df["date_hour"] + pd.Timedelta(hours=1)

category_excluded_dates = category_featured_df["month"].eq(2) & category_featured_df["day"].isin([14, 15, 25])
category_featured_df = category_featured_df.loc[~category_excluded_dates].reset_index(drop=True)
category_featured_df = category_featured_df.sort_values("date_hour").reset_index(drop=True)

category_columns = [col for col in category_featured_df.columns if col.startswith("category_")]
category_feature_cols = [
    "count",
    "hour_sin",
    "hour_cos",
    "is_weekend",
    *day_columns,
    *category_columns,
]

category_model_df = category_featured_df.dropna(
    subset=category_feature_cols + ["predict_1h"]
).reset_index(drop=True)

display(category_model_df[["date_hour", "target_hour", "category", "count", "predict_1h"]].head())

## Загрузка или обучение RandomForest-моделей

In [ ]:
def train_or_load_rf(
    model_path: Path,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    model_label: str,
) -> RandomForestRegressor:
    if USE_SAVED_MODELS and model_path.exists():
        print(f"Загружаю {model_label}: {model_path}")
        return joblib.load(model_path)

    print(f"Обучаю {model_label} и сохраняю в {model_path}")
    model = RandomForestRegressor(
        n_estimators=RF_TREES,
        max_depth=20,
        min_samples_split=20,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    joblib.dump(model, model_path)
    return model


def train_test_split_by_time(model_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    split_index = int(len(model_df) * (1 - TEST_SIZE))
    return model_df.iloc[:split_index].copy(), model_df.iloc[split_index:].copy()


total_train_df, total_test_df = train_test_split_by_time(total_model_df)
category_train_df, category_test_df = train_test_split_by_time(category_model_df)

total_rf_model = train_or_load_rf(
    TOTAL_MODEL_PATH,
    total_train_df[total_feature_cols],
    total_train_df["predict_1h"],
    "RF общего спроса",
)

category_rf_model = train_or_load_rf(
    CATEGORY_MODEL_PATH,
    category_train_df[category_feature_cols],
    category_train_df["predict_1h"],
    "RF категорий",
)

## Production-функция

Функция принимает две модели и сырой датасет, строит признаки внутри и возвращает одну строку результата: общий прогноз и словарь `категория -> процент`.

In [ ]:
def prepare_events_dataset(dataset: pd.DataFrame | str | Path) -> pd.DataFrame:
    if isinstance(dataset, (str, Path)):
        result = pd.read_csv(resolve_data_path(dataset))
    else:
        result = dataset.copy()

    if "date_hour" not in result.columns:
        result["model_response_timestamp"] = pd.to_datetime(
            result["model_response_timestamp"],
            unit="s",
        )
        result["date_hour"] = result["model_response_timestamp"].dt.floor("h")

    result["category"] = result["category"].fillna("unknown").astype(str)
    return result


def align_model_features(df: pd.DataFrame, model, fallback_feature_cols: list[str]) -> pd.DataFrame:
    feature_cols = list(getattr(model, "feature_names_in_", fallback_feature_cols))
    result = df.copy()

    for col in feature_cols:
        if col not in result.columns:
            result[col] = 0

    return result[feature_cols]


def predict_category_percentages_row(
    total_model,
    category_model,
    dataset: pd.DataFrame | str | Path,
    city_name: str = CITY_NAME,
    current_hour=None,
) -> pd.Series:
    prod_events = prepare_events_dataset(dataset)

    prod_total_base = build_total_base(prod_events, city_name)
    prod_total_features = build_total_features(prod_total_base)

    prod_category_base = build_category_base(prod_events, city_name)
    prod_category_features = build_category_features(prod_category_base)

    if current_hour is None:
        current_hour = min(
            prod_total_features["date_hour"].max(),
            prod_category_features["date_hour"].max(),
        )
    current_hour = pd.Timestamp(current_hour).floor("h")

    total_row = prod_total_features.loc[prod_total_features["date_hour"] == current_hour]
    if total_row.empty:
        raise ValueError(f"Нет строки общего спроса для current_hour={current_hour}")

    category_rows = prod_category_features.loc[
        prod_category_features["date_hour"] == current_hour
    ].copy()
    if category_rows.empty:
        raise ValueError(f"Нет строк категорий для current_hour={current_hour}")

    total_X = align_model_features(total_row, total_model, total_feature_cols)
    category_X = align_model_features(category_rows, category_model, category_feature_cols)

    total_count = float(total_model.predict(total_X)[0])
    total_count = max(total_count, 0.0)

    category_counts = category_model.predict(category_X).astype(float)
    category_counts = np.clip(category_counts, 0, None)

    category_sum = category_counts.sum()
    if category_sum > 0:
        category_percentages = category_counts / category_sum
    else:
        category_percentages = np.full(len(category_counts), 1 / len(category_counts))

    percentages_dict = {
        category: float(percent)
        for category, percent in zip(category_rows["category"].values, category_percentages)
    }

    return pd.Series({
        "current_hour": current_hour,
        "target_hour": current_hour + pd.Timedelta(hours=1),
        "total_count": total_count,
        "category_percentages": percentages_dict,
    })


production_row = predict_category_percentages_row(
    total_rf_model,
    category_rf_model,
    events,
)
display(production_row)
print(
    f"total_count={production_row['total_count']:.2f}; "
    f"category_percentages={production_row['category_percentages']}"
)

## Проверка на тестовом периоде

In [ ]:
total_test_pred = total_test_df[["date_hour", "target_hour", "predict_1h"]].copy()
total_test_pred["total_rf_pred"] = np.clip(
    total_rf_model.predict(total_test_df[total_feature_cols]),
    0,
    None,
)

category_test_pred = category_test_df[["date_hour", "target_hour", "category", "predict_1h"]].copy()
category_test_pred["category_model_count"] = np.clip(
    category_rf_model.predict(category_test_df[category_feature_cols]),
    0,
    None,
)

category_test_pred["category_pred_sum"] = category_test_pred.groupby("target_hour")["category_model_count"].transform("sum")
category_test_pred["category_percent"] = np.where(
    category_test_pred["category_pred_sum"] > 0,
    category_test_pred["category_model_count"] / category_test_pred["category_pred_sum"],
    0,
)

combined_test_pred = category_test_pred.merge(
    total_test_pred[["target_hour", "total_rf_pred"]],
    on="target_hour",
    how="inner",
)
combined_test_pred["allocated_total_rf_count"] = (
    combined_test_pred["total_rf_pred"] * combined_test_pred["category_percent"]
)

rmse = np.sqrt(mean_squared_error(
    combined_test_pred["predict_1h"],
    combined_test_pred["allocated_total_rf_count"],
))
mae = mean_absolute_error(
    combined_test_pred["predict_1h"],
    combined_test_pred["allocated_total_rf_count"],
)
r2 = r2_score(
    combined_test_pred["predict_1h"],
    combined_test_pred["allocated_total_rf_count"],
)

metrics_df = pd.DataFrame([
    {"model": "total_rf_scaled_by_category_rf_percent", "RMSE": rmse, "MAE": mae, "R2": r2}
])
display(metrics_df)
display(combined_test_pred.head())

## Визуализация долей последнего прогноза

In [ ]:
latest_distribution = pd.DataFrame({
    "category": list(production_row["category_percentages"].keys()),
    "category_percent": list(production_row["category_percentages"].values()),
})
latest_distribution["target_hour"] = production_row["target_hour"]
plot_df = latest_distribution.sort_values("category_percent", ascending=True)

ax = plot_df.plot.barh(
    x="category",
    y="category_percent",
    legend=False,
    figsize=(12, max(4, len(plot_df) * 0.35)),
)
ax.set_title(f"Доли категорий на {plot_df['target_hour'].iloc[0]}")
ax.set_xlabel("Доля")
ax.set_ylabel("Категория")
ax.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
plt.tight_layout()
plt.show()